In [1]:
'''
Bibliotecas usadas
'''

import os
import re
import subprocess
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from Bio import Entrez
from Bio import SeqIO
from Bio.Seq import Seq
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    make_scorer,
    matthews_corrcoef,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import subprocess
import shap
import seaborn as sns
from sklearn.inspection import permutation_importance


warnings.filterwarnings("ignore")

print("Bibliotecas importadas")


Bibliotecas importadas


In [2]:
'''
obtenção dos datasets
'''

EMAIL = "marcela.leite@gmail.com.br"
OUTPUT_DIR = "pipelineY"
os.makedirs(OUTPUT_DIR, exist_ok=True)
Entrez.email = EMAIL

# ==========================================================
# QUERIES
# ==========================================================

# POSITIVE_QUERY = r'''
# ("Solanum lycopersicum"[Organism] OR "Nicotiana benthamiana"[Organism] OR "Capsicum annuum"[Organism] OR "Arabidopsis thaliana"[Organism])
# AND( TMV OR ToMV OR ToBRFV OR PMMoV OR TMGMV OR PVY OR PepYMV OR TEV OR PVA OR PVX OR TSWV OR GRSV  OR TCSV  OR CMV OR ToCV OR TICV OR AMV
# OR "virus resistance" OR "antiviral" OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" OR "R gene" OR "Sw-5" OR "Tm-2" OR "RCY1"
# OR "Rx" OR "R protein"  OR "NLR" OR "NBS-LRR"  OR "NB-LRR" OR "TIR-NBS-LRR" OR "CC-NBS-LRR" OR "RLK" OR "RLP" OR "LRR receptor kinase"
# OR "leucine rich repeat receptor kinase" OR "plant disease resistance protein" OR "disease resistance protein" OR "RPM1" OR "RPS2" OR "RPS4" OR "RPS5"
# OR "RPP" OR "SNC1"  OR "ADR1" OR "NDR1" OR "EDS1" OR "PAD4" )
# NOT ( partial OR fragment OR hypothetical OR predicted OR DAP-seq OR "transcription factor" OR DNA-binding )
# '''

POSITIVE_QUERY = r'''
    ("Solanaceae"[Organism] NOT ("Solanum betaceum"[Organism] OR "Solanum melongena"[Organism] OR "Solanum tuberosum"[Organism] OR "Physalis peruviana"[Organism])) 
    AND (TMV OR ToMV OR ToBRFV OR PMMoV OR TMGMV OR PVY OR PepYMV OR TEV OR PVA OR PVX OR TSWV OR GRSV OR TCSV OR CMV OR ToCV OR TICV OR AMV
    OR "virus resistance" OR "antiviral" OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" OR "R gene" OR "Sw-5" OR "Tm-2" OR "RCY1" 
    OR "Rx" OR "R protein" OR "NLR" OR "NBS-LRR" OR "NB-LRR" OR "TIR-NBS-LRR" OR "CC-NBS-LRR" OR "RLK" OR "RLP" OR "LRR receptor kinase" 
    OR "leucine rich repeat receptor kinase" OR "plant disease resistance protein" OR "disease resistance protein" OR "RPM1" OR "RPS2" OR "RPS4" OR "RPS5" 
    OR "RPP" OR "SNC1" OR "ADR1" OR "NDR1" OR "EDS1" OR "PAD4" OR "NB-ARC" OR "NBS-ARC" OR "TIR domain"
    OR "NB-ARC domain" OR "NBS-ARC" OR "leucine-rich repeat domain" OR "TIR domain-containing" OR "LRR-containing" OR "coiled-coil leucine-rich repeat")
    NOT (partial OR fragment OR hypothetical OR predicted OR DAP-seq OR "transcription factor" OR DNA-binding OR microtom )    
'''
# #OR microtom 
# NEGATIVE_QUERY = r'''
# ( "Solanum lycopersicum"[Organism] OR "Nicotiana benthamiana"[Organism] OR "Capsicum annuum"[Organism] OR "Arabidopsis thaliana"[Organism])
# AND ( "actin" OR "tubulin" OR "ribosomal protein" OR "ATP synthase" OR "photosystem" OR "elongation factor" OR "glycolysis" OR "metabolic enzyme" )
# NOT ( "virus resistance"  OR "antiviral" OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" OR "R gene" OR "Sw-5" OR "Tm-2"
# OR "RCY1" OR "Rx" OR "R protein" OR "resistance" OR "NLR" OR "LRR" OR "immune" OR "virus" )
# '''
NEGATIVE_QUERY = r'''
    ("Solanaceae"[Organism] NOT ("Solanum melongena"[Organism] OR "Solanum tuberosum"[Organism] OR "Physalis peruviana"[Organism])) 
    AND 100:3000[Sequence Length] 
    NOT (TMV OR ToMV OR ToBRFV OR PMMoV OR TMGMV OR PVY OR PepYMV OR TEV OR PVA OR PVX OR TSWV OR GRSV OR TCSV OR CMV OR ToCV OR TICV OR AMV
    OR "virus resistance" OR "antiviral" OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" OR "R gene" OR "Sw-5" OR "Tm-2" OR "RCY1" 
    OR "Rx" OR "R protein" OR "NLR" OR "NBS-LRR" OR "NB-LRR" OR "TIR-NBS-LRR" OR "CC-NBS-LRR" OR "RLK" OR "RLP" OR "LRR receptor kinase" 
    OR "leucine rich repeat receptor kinase" OR "plant disease resistance protein" OR "disease resistance protein" OR "RPM1" OR "RPS2" OR "RPS4" OR "RPS5" 
    OR "RPP" OR "SNC1" OR "ADR1" OR "NDR1" OR "EDS1" OR "PAD4")
    NOT (partial OR fragment OR hypothetical OR predicted OR uncharacterized OR DAP-seq OR "transcription factor" OR DNA-binding OR microtom )
'''

# Dados dos transcriptomas
ARABIDOPSIS_QUERY = '''
  "Arabidopsis thaliana"[Organism] AND
      (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
      NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
      AND 1500:30000[Sequence Length]
      '''

SOLANUM_MELONGENA = '''
    Solanum melongena [Organism]
AND (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
       NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
       AND 1500:30000[Sequence Length]
    '''

TUBEROSUM_QUERY = '''
    Solanum tuberosum [Organism]
AND (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
       NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
       AND 1500:30000[Sequence Length]
    '''

PHYSALIS_QUERY = '''
  "Physalis peruviana"[Organism] AND
      (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
      NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
      AND 1500:30000[Sequence Length]
      '''


# ==========================================================
# DOWNLOAD NCBI
# ==========================================================

def baixar_proteinas_ncbi(query, output_fasta, max_records=15000):
    print("\n================================")
    print("DOWNLOAD NCBI: ",output_fasta)
    print("================================")
    handle = Entrez.esearch(
        db="protein",
        term=query,
        retmax=max_records
    )
    record = Entrez.read(handle)
    ids = record["IdList"]
    print(f"Proteínas encontradas: {len(ids)}")
    if len(ids) == 0:
        return
    handle = Entrez.efetch(
        db="protein",
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    with open(output_fasta, "w") as f:
        f.write(handle.read())
    print(f"Arquivo salvo: {output_fasta}")

def baixar_transcriptoma(output_fasta, query):
    print("\n================================")
    print("DOWNLOAD TRANSCRIPTOMA: ",output_fasta)
    print("================================")
    handle = Entrez.esearch(
        db="nucleotide",
        term=query,
        retmax=10000
    )
    record = Entrez.read(handle)
    ids = record["IdList"]
    print(f"Transcritos encontrados: {len(ids)}")
    handle = Entrez.efetch(
        db="nucleotide",
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    with open(output_fasta, "w") as f:
        f.write(handle.read())
    print(f"Arquivo salvo: {output_fasta}")


#Dados de Entrada - baixar arquivos FASTA das sequências de proteínas
POSITIVE_FASTA = f"{OUTPUT_DIR}/positive.fasta"
NEGATIVE_FASTA = f"{OUTPUT_DIR}/negative.fasta"

baixar_proteinas_ncbi(POSITIVE_QUERY,POSITIVE_FASTA)
baixar_proteinas_ncbi(NEGATIVE_QUERY,NEGATIVE_FASTA)


print("Arquivos salvos na pasta: ",OUTPUT_DIR)
print("============ FIM COLETA DE DADOS TREINO/VALIDAÇÂO ============")



DOWNLOAD NCBI:  pipelineY/positive.fasta
Proteínas encontradas: 10000
Arquivo salvo: pipelineY/positive.fasta

DOWNLOAD NCBI:  pipelineY/negative.fasta
Proteínas encontradas: 10000
Arquivo salvo: pipelineY/negative.fasta
Arquivos salvos na pasta:  pipelineY
============ FIM COLETA DE DADOS TREINO/VALIDAÇÂO ============


In [3]:
#Dados para aplicação - baixar transcriptomas - RNA-Seq
transcriptoma_arabidopsis_fasta = (f"{OUTPUT_DIR}/arabidopsis_transcriptoma.fasta")
transcriptoma_melongena_fasta = (f"{OUTPUT_DIR}/melongena_transcriptoma.fasta")
transcriptoma_tuberosum_fasta = (f"{OUTPUT_DIR}/tuberosum_transcriptoma.fasta")
transcriptoma_physalis_fasta = (f"{OUTPUT_DIR}/physalis_transcriptoma.fasta")

baixar_transcriptoma(transcriptoma_arabidopsis_fasta, ARABIDOPSIS_QUERY)
baixar_transcriptoma(transcriptoma_melongena_fasta, SOLANUM_MELONGENA)
baixar_transcriptoma(transcriptoma_tuberosum_fasta, TUBEROSUM_QUERY)
baixar_transcriptoma(transcriptoma_physalis_fasta, PHYSALIS_QUERY)
print("Arquivos salvos na pasta: ",OUTPUT_DIR)
print("============ FIM COLETA DE DADOS APLICAÇÃO ============")


DOWNLOAD TRANSCRIPTOMA:  pipelineY/arabidopsis_transcriptoma.fasta
Transcritos encontrados: 4453
Arquivo salvo: pipelineY/arabidopsis_transcriptoma.fasta

DOWNLOAD TRANSCRIPTOMA:  pipelineY/melongena_transcriptoma.fasta
Transcritos encontrados: 8589
Arquivo salvo: pipelineY/melongena_transcriptoma.fasta

DOWNLOAD TRANSCRIPTOMA:  pipelineY/tuberosum_transcriptoma.fasta
Transcritos encontrados: 300
Arquivo salvo: pipelineY/tuberosum_transcriptoma.fasta

DOWNLOAD TRANSCRIPTOMA:  pipelineY/physalis_transcriptoma.fasta
Transcritos encontrados: 10000
Arquivo salvo: pipelineY/physalis_transcriptoma.fasta
Arquivos salvos na pasta:  pipelineY
============ FIM COLETA DE DADOS APLICAÇÃO ============


In [4]:
'''
Tratamento dos dados
    - Criar Dataframes com os conjuntos de dados de treino e validação
    - Executar CD-Hit para eliminar sequencias muitos semelhantes
    - Extração de atributos (features)

'''

# ==========================================================
# AAC
# ==========================================================
# composição de aminoácidos - transformar a proteína em dados numéricos
# conta a frequência de cada aminoácido em uma sequẽncia de proteína
# como são 20 aminoácidos possíveis, irá gerar 20 features com seus percentuais
# com essa frequência consegue ter um parâmetro para distinguir as proteínas
# por exemplo proteínas de resistência tem geralmente certa frequência de aminoácidos
# mas ignora a ordem - por isso a sequẽncia de dipeptídeos irá incrementar o modelo considerando a ordem
# sequência de 20 aminoácidos
AMINO_ACIDOS = list("ACDEFGHIKLMNPQRSTVWY")

motifs = {
    # "P_LOOP": r"[A-Z]G[A-Z]{4}GK[ST]",
    # "GLPL": r"GLPL",
    # "MHD": r"MHD"

    # # novos motifs
    # "KINASE2": r"[LIVM]{3,4}DD[VLIM]",
    # "RNBS_A": r"FDLxKxWVSVSD",
    # "RNBS_B": r"[A-Z]{2}G[PH][A-Z]{2}",
    # "RNBS_C": r"[A-Z]{2}P[A-Z]{2}W",
    # "RNBS_D": r"CFLYCALFPED",
    # "VVVLDDVW": r"VVVLDDVW"
    # "P_LOOP": r"[A-Z]G[A-Z]{4}GK[ST]",
    # "P_LOOP": r"G[PAST][A-Z]{4}GK[ST]",
    "P_LOOP": r"G[A-Z]{4}GK[ST]",
    # "KINASE2": r"[LIVM]{3,5}DD",
    "KINASE2": r"[LIVM]{4}DD[VIL]",
    "RNBS_A": r"FDL",
    "RNBS_B": r"G[RK]G",
    "RNBS_C": r"[A-Z]{2}P[A-Z]{2}W",
    "GLPL": r"GLPL",
    "RNBS_D": r"CFL",
    "MHD": r"MHD"
}

def calcular_aac(seq):
    seq = seq.upper()
    total = len(seq)
    if total == 0:
        return [0] * len(AMINO_ACIDOS)
    counts = Counter(seq)
    return [
        counts.get(aa, 0) / total
        for aa in AMINO_ACIDOS
    ]

# ==========================================================
# DIPEPTÍDEOS
# ==========================================================
'''
encontrar pares de aminoácidos consectíveis em uma sequência protéica  -sequência de dois aminoácidos
considera a ordem dos aminoácidos
há certos padrões comuns para 
para 20 aminoácidos, há possibilidade de 20*20 = 400 dipeptídeos - features
'''

DIPEPTIDES = [
    a + b
    for a in AMINO_ACIDOS
    for b in AMINO_ACIDOS
]

def calcular_dipeptideos(seq):
    seq = seq.upper()
    total = len(seq) - 1
    if total <= 0:
        return [0] * len(DIPEPTIDES)
    counts = Counter([
        seq[i:i+2]
        for i in range(total)
    ])
    return [
        counts.get(dp, 0) / total
        for dp in DIPEPTIDES
    ]

def localizar_motivos(seq):
    posicoes = {}
    for nome, pattern in motifs.items():
        match = re.search(pattern, seq)

        if match:
            posicoes[nome] = match.start()
        else:
            posicoes[nome] = None

    return posicoes
    

def calcular_distancias(posicoes):

    def distancia(a, b):
        if posicoes[a] is None or posicoes[b] is None:
            return -1
        return posicoes[b] - posicoes[a]

    return {
        "dist_ploop_kinase2": distancia("P_LOOP", "KINASE2"),
        "dist_kinase2_glpl": distancia("KINASE2", "GLPL"),
        "dist_glpl_mhd": distancia("GLPL", "MHD")
    }
    
def contar_motivos(posicoes):
    return sum(
        1 for v in posicoes.values()
        if v is not None
    )
    
def extrair_motifs(seq):

    return [
        int(bool(re.search(pattern, seq)))
        for pattern in motifs.values()
    ]



# ==========================================================
# FEATURES - atributos
# ==========================================================
#calculando features
def extrair_features_proteina(seq):
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    if len(seq) < 50:
        return None
    features = []
    features.append(len(seq)) # 1 - comprimento da cadeia
    features.extend(calcular_aac(seq)) # 20 composição de aminoácidos - frequência de cada um na sequência
    features.extend(calcular_dipeptideos(seq)) #400 possibilidades - considera a ordem dos aminoácidos para buscar os mais frequêntes em proteínas de resistência
    
    features.extend(extrair_motifs(seq))
    # posições dos motivos
    posicoes = localizar_motivos(seq)
    # número de motivos encontrados
    num_motifs_detectados = contar_motivos(posicoes)
    # distâncias entre motivos
    distancias = calcular_distancias(posicoes)
    
    features.append(num_motifs_detectados)
    protein_length = len(seq)

    features.append(distancias["dist_ploop_kinase2"])
    features.append(distancias["dist_kinase2_glpl"])
    features.append(distancias["dist_glpl_mhd"])
    
    features.append(
        distancias["dist_ploop_kinase2"]/protein_length
        if distancias["dist_ploop_kinase2"] != -1 else -1
    )
    features.append(
        distancias["dist_kinase2_glpl"]/protein_length
        if distancias["dist_kinase2_glpl"] != -1 else -1
    )
    features.append(
        distancias["dist_glpl_mhd"]/protein_length
        if distancias["dist_glpl_mhd"] != -1 else -1
    )
        
    features.append(int(posicoes["P_LOOP"] is not None))
    features.append(int(posicoes["KINASE2"] is not None))
    features.append(int(posicoes["GLPL"] is not None))
    features.append(int(posicoes["MHD"] is not None))
    
    return features
    
# ==========================================================
# DATASET
# ==========================================================
motif_cols = [
    "P_LOOP",
    "KINASE2",
    "RNBS_A",
    "RNBS_B",
    "RNBS_C",
    "GLPL",
    "RNBS_D",
    "MHD"
]

def criar_dataset(positive_fasta, negative_fasta):
    data = []
    columns = (
        ["protein_length"]
        + [f"AAC_{aa}" for aa in AMINO_ACIDOS]
        + [f"DIPEP_{dp}" for dp in DIPEPTIDES]
        + motif_cols
        + ["num_motifs_detectados"]
        + ["dist_ploop_kinase2"]
        + ["dist_kinase2_glpl"]
        + ["dist_glpl_mhd"]
        + ["dist_ploop_kinase2_norm"]
        + ["dist_kinase2_glpl_norm"]
        + ["dist_glpl_mhd_norm"]        
        + ["has_ploop"]
        + ["has_kinase2"]
        + ["has_glpl"]
        + ["has_mhd"]
    )

    # POSITIVOS
    for record in SeqIO.parse(positive_fasta, "fasta"):
        seq = str(record.seq)
        feats = extrair_features_proteina(seq)
        if feats is None:
            continue
        row = feats + [1, record.id]
        data.append(row)

    # NEGATIVOS
    for record in SeqIO.parse(negative_fasta, "fasta"):
        seq = str(record.seq)
        feats = extrair_features_proteina(seq)
        if feats is None:
            continue
        row = feats + [0, record.id]
        data.append(row)
        
    df = pd.DataFrame(
        data,
        columns=columns + ["target", "id"]
    )
    print("\nTotal de Features/Atributos:",len(columns)) # quantidade
    print("\nFeatures/atributos:")
    print(columns)     
    return df, columns

# CD-Hit
# chama o programa CD-HIT do sistema operacional
def executar_cdhit(
    input_fasta,
    output_fasta,
    identity=0.7  #0.5
):
    cmd = [
        "cd-hit",
        "-i", input_fasta,
        "-o", output_fasta,
        "-c", str(identity),
        "-n", "5",
        "-M", "30000",
        "-T", "4"
    ]

    print("\nExecutando CD-HIT...")
    subprocess.run(cmd, check=True)
    print("CD-HIT concluído.")

def limpar(seq_id):
    seq_id = seq_id.strip()
    seq_id = seq_id.split()[0]
    return seq_id

def ler_clusters_cdhit(cluster_file):
    clusters = {}
    cluster_id = None
    with open(cluster_file) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">Cluster"):
                cluster_id = line.replace(">Cluster ", "")
                clusters[cluster_id] = []
            else:
                seq_id = line.split(">")[1].split("...")[0]
                seq_id = limpar(seq_id)
                clusters[cluster_id].append(seq_id)
    return clusters

# executa o CD-Hit nos arquivos baixados
executar_cdhit(
    POSITIVE_FASTA,
    f"{OUTPUT_DIR}/positive_nr.fasta",
    identity=0.7
)

executar_cdhit(
    NEGATIVE_FASTA,
    f"{OUTPUT_DIR}/negative_nr.fasta",
    identity=0.7
)
# ======================================================
# Montar os DATASETs
# ======================================================

POSITIVE_FASTA = f"{OUTPUT_DIR}/positive_nr.fasta"
NEGATIVE_FASTA = f"{OUTPUT_DIR}/negative_nr.fasta"


df, columns = criar_dataset(
    POSITIVE_FASTA,
    NEGATIVE_FASTA
)

df["id"] = df["id"].apply(limpar)
# ======================================================
# CLUSTERS
# ======================================================

clusters_pos = ler_clusters_cdhit(
    f"{OUTPUT_DIR}/positive_nr.fasta.clstr"
)

clusters_neg = ler_clusters_cdhit(
    f"{OUTPUT_DIR}/negative_nr.fasta.clstr"
)

cluster_map = {}

for c, seqs in clusters_pos.items():
    for s in seqs:
        cluster_map[s] = f"POS_{c}"

for c, seqs in clusters_neg.items():
    for s in seqs:
        cluster_map[s] = f"NEG_{c}"

df["cluster"] = df["id"].map(cluster_map)

print("\n Clusters ausentes: ")
print(df["cluster"].isna().sum())
df = df.dropna(subset=["cluster"])

print("\nDataset original pós CD-HIT:", df.shape)

# ======================================================
# NOVO BLOCO: BALANCEAMENTO IMPERATIVO (DOWNSAMPLING)
# ======================================================
print("\n--- Executando Balanceamento de Classes ---")
df_positivos = df[df["target"] == 1]
df_negativos = df[df["target"] == 0]

print(f"Contagem inicial -> Positivos (R-genes): {len(df_positivos)} | Negativos (Background): {len(df_negativos)}")

# Se houver mais negativos do que positivos (cenário padrão), reduzimos os negativos
if len(df_negativos) > len(df_positivos):
    # O sample() escolhe aleatoriamente N linhas do grupo majoritário
    df_negativos_balanceados = df_negativos.sample(n=len(df_positivos), random_state=42)
    df = pd.concat([df_positivos, df_negativos_balanceados]).reset_index(drop=True)
    print(f"Subamostragem aplicada com sucesso nos Negativos.")
else:
    print("O dataset já está balanceado ou possui mais positivos que negativos (incomum). Nenhuma ação foi tomada.")

print(f"Contagem final   -> Positivos: {len(df[df['target'] == 1])} | Negativos: {len(df[df['target'] == 0])}")
print("Dataset Final Pronto:", df.shape)

print("Fim tratamento dos dados...")




Executando CD-HIT...
Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i pipelineY/positive.fasta -o
         pipelineY/positive_nr.fasta -c 0.7 -n 5 -M 30000 -T 4

Started: Tue Jun  9 23:49:10 2026
                            Output                              
----------------------------------------------------------------
total seq: 9999
longest and shortest : 3540 and 29
Total letters: 7344091
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 8M
Buffer          : 4 X 11M = 45M
Table           : 2 X 65M = 131M
Miscellaneous   : 0M
Total           : 185M

Table limit with the given memory limit:
Max number of representatives: 4000000
Max number of word counting entries: 3726846984

# comparing sequences from          0  to       1666
.---------- new table with      253 representatives
# comparing sequences from       1666  to       3054
----------    502 remaining sequences to the next cycle
---------- new table with    

In [5]:
'''
k-Fold-Cross-Validation

'''

def validacao_cruzada(nome, model, X_train, y_train, groups_train):
    
    cv = StratifiedGroupKFold(
        n_splits=5, #10
        shuffle=True,
        random_state=42
    )
    mcc_scorer = make_scorer(matthews_corrcoef)
    scoring_dict = {
        'recall': 'recall',
        'roc_auc': 'roc_auc',
        'mcc': mcc_scorer,
        'accuracy': 'accuracy',
        'precision': 'precision',
        'f1': 'f1'
    }
    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=scoring_dict,
        n_jobs=-1
    )
    print(f"Finalizada a validação cruzada para: {nome}")
    return scores  
  

In [6]:
'''
Criação dos modelos/ Treinamento
'''
# para melhor separar os conjuntos de treino e teste considerando os agrupamentos feitos no CD-HIT para sequẽncias muito parecidas
def split_por_homologia(df):
    print(df.head(30))
    groups = df["cluster"]
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )
    # cv  = StratifiedGroupKFold(
    #     n_splits = 10,
    #     shuffle=True,
    #     random_state = 42
    # )    

    train_idx, test_idx = next(
        splitter.split(df, groups=groups)
        # cv.split(df, df["target"], groups)
    )
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]
    return train_df, test_df

def calcular_e_salvar_shap(nome, model, X_test, feature_names, output_dir):
    """
    Calcula os SHAP values para modelos de árvore e salva os gráficos explicativos.
    """
    print(f"\n[SHAP] Inicializando análise de interpretabilidade para: {nome}...")
    
    # 1. Inicializa o explicador otimizado para árvores
    explainer = shap.TreeExplainer(model)
    
    # 2. Calcula os SHAP values para o conjunto de teste
    shap_values = explainer.shap_values(X_test)
    
    # 3. Tratamento de formato (Scikit-Learn Random Forest vs XGBoost)
    # O Random Forest do sklearn retorna uma lista contendo [valores_classe_0, valores_classe_1]
    # O XGBoost retorna diretamente a matriz de SHAP values para a classe alvo
    if isinstance(shap_values, list):
        # Selecionamos o índice 1 (classe positiva: proteínas de resistência/alvos)
        shap_values_pos = shap_values[1]
    elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
        # Tratamento para versões específicas do SHAP que geram arrays 3D
        shap_values_pos = shap_values[:, :, 1]
    else:
        shap_values_pos = shap_values

    # 4. Gráfico 1: Summary Plot (Beeswarm)
    # Mostra o impacto biológico de cada feature (ex: se alta frequência aumenta ou diminui a predição)
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_pos, 
        X_test, 
        feature_names=feature_names, 
        max_display=20,  # Foco nas top 20 características
        show=False
    )
    # plt.title(f"SHAP Summary Plot (Top 20) - {nome}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{nome}_shap_summary.png", bbox_inches="tight", dpi=300)
    plt.close()
    
    # 5. Gráfico 2: Bar Plot (Importância Global)
    # Mostra a magnitude média do impacto de cada feature de forma absoluta
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_pos, 
        X_test, 
        feature_names=feature_names, 
        plot_type="bar", 
        max_display=20, 
        show=False
    )
    # plt.title(f"SHAP Global Feature Importance - {nome}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{nome}_shap_bar.png", bbox_inches="tight", dpi=300)
    plt.close()
    
    print(f"[SHAP] Gráficos salvos com sucesso em '{output_dir}/'!")


def calcular_shap_svm(nome, model, X_test, y_test, output_dir):
    # ======================================================
    # IMPORTÂNCIA POR PERMUTAÇÃO PARA O SVM (Novo)
    # ======================================================
    print("\n[SVM] Calculando a importância por permutação dos atributos...")
    
    # Calcula a permutação no X_test (pode usar X_train se quiser focar no ajuste ao treino)
    result_svm = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
    
    # Monta o DataFrame com os resultados
    importancia_svm = pd.DataFrame({
        'feature': columns,
        'importance_mean': result_svm.importances_mean
    }).sort_values('importance_mean', ascending=False)
    
    print("\nTop 20 features mais importantes para o SVM:")
    print(importancia_svm.head(20))
    
    # Gera o gráfico de barras igual ao do XGBoost
    top_svm = importancia_svm.head(20)
    plt.figure(figsize=(10,8))
    plt.barh(top_svm["feature"][::-1], top_svm["importance_mean"][::-1], color='seagreen')
    plt.xlabel("Queda no Desempenho (Média)")
    plt.ylabel("Feature / Atributo")
    # plt.title("Top 20 Features - SVM Permutation Importance")
    plt.tight_layout()
    plt.savefig(f"{output_dir}/svm_importancia_permutacao.png", bbox_inches="tight", dpi=300)
    plt.close()
    print(f"[SVM] Gráfico salvo com sucesso em '{output_dir}/svm_importancia_permutacao.png'!")
    
# ==========================================================
# TREINAMENTO - treina os modelos
# ==========================================================

def avaliar_modelo(nome, model, X_train, y_train, X_test, y_test):
    print("\n============================================================")
    print(nome)
    print("============================================================")
    model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:,1]
    preds = (probs >= 0.5).astype(int)
    print(classification_report(y_test, preds))
    roc = roc_auc_score(y_test, probs)
    pr = average_precision_score(y_test, probs)
    mcc = matthews_corrcoef(y_test, preds)
    f1 = f1_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    bal_acc = balanced_accuracy_score(y_test, preds)

    print("ROC-AUC          :", roc)
    print("PR-AUC           :", pr)
    print("MCC              :", mcc)
    print("F1               :", f1)
    print("Precision        :", precision)
    print("Recall           :", recall)
    print("Balanced Accuracy:", bal_acc)

    # ======================================================
    # MATRIZ DE CONFUSÃO
    # ======================================================
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5,5))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Negativo", "Positivo"]
    )
    disp.plot(cmap="Blues")
    # plt.title(f"Matriz de Confusão - {nome}")
    plt.savefig(
        f"{OUTPUT_DIR}/{nome}_matriz_confusao.png",
        bbox_inches="tight"
    )
    plt.close()
    # ======================================================
    # CURVA ROC
    # ======================================================
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6,6))
    plt.plot(
        fpr,
        tpr,
        label=f"AUC = {roc_auc:.4f}"
    )
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    # plt.title(f"Curva ROC - {nome}")
    plt.legend(loc="lower right")
    plt.savefig(
        f"{OUTPUT_DIR}/{nome}_roc.png",
        bbox_inches="tight"
    )
    plt.close()
    
    return {
        "Modelo": nome,
        "ROC_AUC": roc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "Balanced_Accuracy": bal_acc,
        "MCC":mcc
    }, model


# ======================================================
# Separação Treino/Teste 80/20
# SPLIT POR HOMOLOGIA
# ======================================================

train_df, test_df = split_por_homologia(df)
X_train = train_df[columns]
y_train = train_df["target"]
X_test = test_df[columns]
y_test = test_df["target"]
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ======================================================
# MODELOS
# ======================================================

rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth = 12,
    min_samples_split=4,
    max_features='sqrt',
    random_state=42,
    class_weight="balanced",
    n_jobs = -1
)    
xgb = XGBClassifier(
    n_estimators=1200,
    max_depth=5,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    min_child_weight=3,
    reg_alpha=0.5,
    reg_lambda=2,
    scale_pos_weight=1.2
)
svm = SVC(
    probability=True,
    kernel="rbf",
    class_weight="balanced",
    random_state=42,
    C=2,
    gamma='scale'
)
scores = []
groups_train = train_df["cluster"]

scores_rf = validacao_cruzada('RF',rf,X_train, y_train, groups_train)
scores_xgb = validacao_cruzada('XGB',xgb, X_train, y_train, groups_train)
scores_svm = validacao_cruzada('SVM',svm, X_train, y_train, groups_train)

print("Cross-Validation:RF")
for k,v in scores_rf.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")

print("Cross-Validation XGB")
for k,v in scores_xgb.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")

print("Cross-Validation SVM")
for k,v in scores_svm.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")


resultados_cv = pd.DataFrame({
    'Modelo': ['RF','XGB','SVM'],
    'Accuracy': [
        scores_rf['test_accuracy'].mean(),
        scores_xgb['test_accuracy'].mean(),
        scores_svm['test_accuracy'].mean(),
    ],    
    'Recall': [
        scores_rf['test_recall'].mean(),
        scores_xgb['test_recall'].mean(),
        scores_svm['test_recall'].mean(),
    ],
    'Precision': [
        scores_rf['test_precision'].mean(),
        scores_xgb['test_precision'].mean(),
        scores_svm['test_precision'].mean(),
    ],
    'F1': [
        scores_rf['test_f1'].mean(),
        scores_xgb['test_f1'].mean(),
        scores_svm['test_f1'].mean(),
    ],
    'ROC_AUC': [
        scores_rf['test_roc_auc'].mean(),
        scores_xgb['test_roc_auc'].mean(),
        scores_svm['test_roc_auc'].mean(),
    ],
    'MCC': [
        scores_rf['test_mcc'].mean(),
        scores_xgb['test_mcc'].mean(),
        scores_svm['test_mcc'].mean(),
    ]
})
print(resultados_cv)
resultados_cv.to_csv(
        f"{OUTPUT_DIR}/metricas_cv.csv",
        index=False
    )
dados = []

for score in scores_rf['test_roc_auc']:
    dados.append(['RF',score])
for score in scores_xgb['test_roc_auc']:
    dados.append(['XGB',score])
for score in scores_svm['test_roc_auc']:
    dados.append(['SVM',score])

grafico = pd.DataFrame(dados, columns=['Modelo','ROC_AUC'])
grafico.boxplot(by='Modelo')
plt.ylabel("ROC-AUC")
# plt.title("10-Fold Cross Validation")
plt.suptitle("")
plt.tight_layout()
# plt.show()
plt.savefig(
    f"{OUTPUT_DIR}/kfold_cross_validation_roc.png",
    bbox_inches="tight"
)
plt.close()



################################################
# Treinamento

results = []
r1, rf_model = avaliar_modelo(
    "Random Forest",
    rf,
    X_train,
    y_train,
    X_test,
    y_test
) 
results.append(r1)
calcular_e_salvar_shap("Random Forest", rf_model, X_test, columns,OUTPUT_DIR )

r2, xgb_model = avaliar_modelo(
    "XGBoost",
    xgb,
    X_train,
    y_train,
    X_test,
    y_test
)
results.append(r2)
calcular_e_salvar_shap("XGBoost", xgb_model, X_test, columns,OUTPUT_DIR )

r3, svm_model = avaliar_modelo(
    "SVM",
    svm,
    X_train,
    y_train,
    X_test,
    y_test
)
results.append(r3)
calcular_shap_svm("SVM", svm_model, X_test, y_test, OUTPUT_DIR)

print("\nFim criação/treino dos modelos\n")


    protein_length     AAC_A     AAC_C     AAC_D     AAC_E     AAC_F  \
0             1053  0.034188  0.018993  0.043685  0.038936  0.058879   
1              231  0.090909  0.012987  0.069264  0.077922  0.051948   
2              201  0.039801  0.004975  0.034826  0.044776  0.019900   
3              888  0.055180  0.018018  0.061937  0.072072  0.045045   
4              504  0.103175  0.019841  0.045635  0.071429  0.029762   
5              704  0.049716  0.007102  0.080966  0.117898  0.041193   
6              535  0.095327  0.009346  0.020561  0.026168  0.082243   
7              450  0.040000  0.028889  0.035556  0.082222  0.024444   
18             971  0.045314  0.017508  0.054583  0.071061  0.036045   
19             666  0.039039  0.016517  0.057057  0.057057  0.040541   
20             709  0.053597  0.014104  0.049365  0.071932  0.043724   
21             999  0.053053  0.018018  0.070070  0.050050  0.036036   
22            1071  0.056956  0.020542  0.059757  0.038282  0.04

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

In [7]:
'''
Avaliação e Validação dos modelos
'''
# verificar importância dos atributos/feature para o Random Forest
importancia = pd.DataFrame({"feature":columns,"importance":rf_model.feature_importances_})
importancia = importancia.sort_values("importance",ascending = False)
print("\nTop 20 features: Random Forest")
print(importancia.head(20))
# GRÁFICO
top = importancia.head(20)
plt.figure(figsize=(10,8))
plt.barh( top["feature"][::-1], top["importance"][::-1] )
plt.xlabel("Importância")
plt.ylabel("Feature")
# plt.title("Top 20 Features - Random Forest")
plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/rf_importancia_variaveis.png",
    bbox_inches="tight"
)
plt.close()

# verificar importância dos atributos/feature para o XGBoost
importancia = pd.DataFrame({"feature":columns, "importance":xgb_model.feature_importances_})
importancia = importancia.sort_values("importance", ascending = False)
print("\nTop 20 features: XGBoost")
print(importancia.head(20))
# GRÁFICO
top = importancia.head(20)
plt.figure(figsize=(10,8))
plt.barh(top["feature"][::-1], top["importance"][::-1])
plt.xlabel("Importância")
plt.ylabel("Feature")
# plt.title("Top 20 Features - XGBoost")
plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/xgb_importancia_variaveis.png",
    bbox_inches="tight"
)
plt.close()

print("\nCOMPARAÇÃO MODELOS")
print(pd.DataFrame(results))

results_df = pd.DataFrame(results)
results_df.to_csv(
        f"{OUTPUT_DIR}/metricas_comparacao.csv",
        index=False
    )

metrics = [
    "ROC_AUC",
    "F1",
    "Precision",
    "Recall",
    "Balanced_Accuracy"
]

for metric in metrics:
    plt.figure(figsize=(8,5))
    plt.bar(
        results_df["Modelo"],
        results_df[metric]
    )

    plt.ylim(0, 1)
    plt.ylabel(metric)
    # plt.title(f"Comparação dos Modelos - {metric}")
    for i, v in enumerate(results_df[metric]):
        plt.text(i, v + 0.01, f"{v:.3f}", ha='center')

    plt.savefig(
        f"{OUTPUT_DIR}/comparacao_{metric}.png",
        bbox_inches="tight"
    )
    plt.close()

modelos = {
    "Random Forest":rf_model,
    "XGBoost":xgb_model,
    "SVM":svm_model
    }
plt.figure(figsize=(7,7))
for nome, model in modelos.items():
    probs = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(
        fpr,
        tpr,
        label=f"{nome} AUC={roc_auc:.3f}"
    )
plt.plot([0,1],[0,1],'--')
plt.xlabel("FPR")
plt.ylabel("TPR")
# plt.title("Comparação ROC")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/roc_comparativa.png")
plt.close()

print("Fim avaliação / validação modelos ")


Top 20 features: Random Forest
                     feature  importance
10                     AAC_L    0.032013
429    num_motifs_detectados    0.031872
1                      AAC_A    0.027443
421                   P_LOOP    0.027375
436                has_ploop    0.025950
437              has_kinase2    0.024482
426                     GLPL    0.022985
438                 has_glpl    0.021822
209                 DIPEP_LK    0.018855
250                 DIPEP_NL    0.018527
422                  KINASE2    0.018360
433  dist_ploop_kinase2_norm    0.015559
0             protein_length    0.015236
430       dist_ploop_kinase2    0.014562
204                 DIPEP_LE    0.013387
215                 DIPEP_LR    0.011270
190                 DIPEP_KL    0.009887
434   dist_kinase2_glpl_norm    0.009061
21                  DIPEP_AA    0.008797
431        dist_kinase2_glpl    0.008525

Top 20 features: XGBoost
                   feature  importance
421                 P_LOOP    0.087432
429

In [8]:
'''
Funções para rodar a aplicação
'''
motifs = {
    "P_LOOP": r"[A-Z]G[A-Z]{4}GK[ST]",
    "KINASE2": r"[LIVM]{3,5}DD",
    "RNBS_A": r"FDL",
    "RNBS_B": r"G[RK]G",
    "RNBS_C": r"[A-Z]{2}P[A-Z]{2}W",
    "GLPL": r"GLPL",
    "RNBS_D": r"CFL",
    "MHD": r"MHD",
}

df_validacao = []

def validar_motifs(df,planta,modelo):
    print("\nVALIDAÇÃO BIOLÓGICA\n")
    counts = {}
    for motif_name, pattern in motifs.items():
        counts[motif_name] = 0
        for seq in df["protein_seq"]:
            if re.search(pattern, seq):
                counts[motif_name] += 1
    for k, v in counts.items():
        print(k, ":", v)

    counts['planta'] = planta
    counts['modelo'] = modelo
    
    df_validacao.append(counts)
    df_counts = pd.DataFrame([counts])
    df_counts.to_csv(
        f"{OUTPUT_DIR}/validacao_biologica_{modelo}_{planta}.csv",
        index=False
    )



def rodar_transdecoder(fasta_nt):
    cmd1 = [
        "TransDecoder.LongOrfs",
        "-t",
        fasta_nt,
        "--output_dir",
        f"{OUTPUT_DIR}"
    ]
    with open(fasta_nt+"_output1.txt", "a", encoding="utf-8") as log_file:
        subprocess.run(cmd1, 
                       stdout=log_file, 
                       stderr=subprocess.STDOUT,
                       text=True)

    cmd2 = [
        "TransDecoder.Predict",
        "-t",
        fasta_nt,
        "--output_dir",
        f"{OUTPUT_DIR}"
    ]
    with open(fasta_nt+"_output2.txt", "a", encoding="utf-8") as log_file:
        subprocess.run(cmd2, 
                       stdout=log_file, 
                       stderr=subprocess.STDOUT,
                       text=True)
    pep_file = fasta_nt + ".transdecoder.pep"
    return pep_file


def predizer_transcritos(fasta_nt,model,nmodel, output, planta):
    probabilidade = 0.85
    candidatos = []
    pep_file = rodar_transdecoder(fasta_nt)
    for record in SeqIO.parse(fasta_nt+".transdecoder.pep", "fasta"):
        protein = str(record.seq)
        feats = extrair_features_proteina(protein)
        if feats is None:
            continue
        X = scaler.transform([feats])
        prob = model.predict_proba(X)[0][1]
        candidatos.append({
            "id": record.id,
            "descricao": record.description,
            "protein_length": len(protein),
            "probabilidade_resistencia": prob,
            "protein_seq": protein
        })

    candidatos_df = pd.DataFrame(candidatos)
    candidatos_df = candidatos_df.sort_values(
        "probabilidade_resistencia",
        ascending=False
    )

    # top = candidatos_df.head(40)
    top = candidatos_df[candidatos_df["probabilidade_resistencia"] >= probabilidade]
    print("\nTOP CANDIDATOS com probabilidade de >= {probabilidade}: "+fasta_nt+", MODELO: "+nmodel)
    print(top[[
        "id",
        "descricao",
        "protein_length",
        "probabilidade_resistencia"
    ]])

    validar_motifs(top,planta,nmodel)

    candidatos_df.to_csv(
        f"{OUTPUT_DIR}/{output}",
        index=False
    )


In [9]:
'''
Aplicação - Modelo XGBoost
'''

print("\nPredição para ARABIDOPSIS\n")
predizer_transcritos(transcriptoma_arabidopsis_fasta, xgb_model,'XGBoost',output="predicoes_arabidopsis_xgboost.csv",planta="Arabidopsis thaliana")

print("\nPredição para MELONGENA\n")
predizer_transcritos(transcriptoma_melongena_fasta, xgb_model,'XGBoost',output="predicoes_melongena_xgboost.csv",planta="Solanum melongena")

print("\nPredição para TUBEROSUM\n")
predizer_transcritos(transcriptoma_tuberosum_fasta, xgb_model,'XGBoost',output="predicoes_tuberosum_xgboost.csv",planta="Solanum tuberosum")

print("\nPredição para PHYSALIS\n")
predizer_transcritos(transcriptoma_physalis_fasta, xgb_model,'XGBoost',output="predicoes_physalis_xgboost.csv",planta="Physalis peruviana")

print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_xgboost.csv")




Predição para ARABIDOPSIS


TOP CANDIDATOS com probabilidade de >= {probabilidade}: pipelineY/arabidopsis_transcriptoma.fasta, MODELO: XGBoost
                 id                                          descricao  \
644   AX507147.1.p1  AX507147.1.p1 GENE.AX507147.1~~AX507147.1.p1  ...   
1896  JB347889.1.p1  JB347889.1.p1 GENE.JB347889.1~~JB347889.1.p1  ...   
2598  LQ603450.1.p1  LQ603450.1.p1 GENE.LQ603450.1~~LQ603450.1.p1  ...   
1857  JB347605.1.p1  JB347605.1.p1 GENE.JB347605.1~~JB347605.1.p1  ...   
2559  LQ603166.1.p1  LQ603166.1.p1 GENE.LQ603166.1~~LQ603166.1.p1  ...   
1870  JB347704.1.p1  JB347704.1.p1 GENE.JB347704.1~~JB347704.1.p1  ...   
1905  JB347983.1.p1  JB347983.1.p1 GENE.JB347983.1~~JB347983.1.p1  ...   
2572  LQ603265.1.p1  LQ603265.1.p1 GENE.LQ603265.1~~LQ603265.1.p1  ...   
2607  LQ603544.1.p1  LQ603544.1.p1 GENE.LQ603544.1~~LQ603544.1.p1  ...   
2633  LQ605013.1.p1  LQ605013.1.p1 GENE.LQ605013.1~~LQ605013.1.p1  ...   
1931  JB349452.1.p1  JB349452.1.p1 GENE.JB

In [10]:
print(df_validacao)

[{'P_LOOP': 32, 'KINASE2': 25, 'RNBS_A': 13, 'RNBS_B': 2, 'RNBS_C': 6, 'GLPL': 15, 'RNBS_D': 15, 'MHD': 17, 'planta': 'Arabidopsis thaliana', 'modelo': 'XGBoost'}, {'P_LOOP': 8, 'KINASE2': 14, 'RNBS_A': 2, 'RNBS_B': 7, 'RNBS_C': 0, 'GLPL': 12, 'RNBS_D': 12, 'MHD': 2, 'planta': 'Solanum melongena', 'modelo': 'XGBoost'}, {'P_LOOP': 8, 'KINASE2': 2, 'RNBS_A': 1, 'RNBS_B': 5, 'RNBS_C': 1, 'GLPL': 7, 'RNBS_D': 8, 'MHD': 2, 'planta': 'Solanum tuberosum', 'modelo': 'XGBoost'}, {'P_LOOP': 52, 'KINASE2': 57, 'RNBS_A': 13, 'RNBS_B': 18, 'RNBS_C': 10, 'GLPL': 44, 'RNBS_D': 39, 'MHD': 17, 'planta': 'Physalis peruviana', 'modelo': 'XGBoost'}]


In [11]:
'''
Aplicação - Modelo Random Forest
'''
print("\nPredição para ARABIDOPSIS\n")
predizer_transcritos(transcriptoma_arabidopsis_fasta, rf_model,"RF",output="predicoes_arabidopsis_rf.csv",planta="Arabidopsis thaliana")

print("\nPredição para MELONGENA\n")
predizer_transcritos(transcriptoma_melongena_fasta, rf_model,'RF',output="predicoes_melongena_rf.csv",planta="Solanum melongena")

print("\nPredição para TUBEROSUM\n")
predizer_transcritos(transcriptoma_tuberosum_fasta, rf_model,"RF",output="predicoes_tuberosum_rf.csv",planta="Solanum tuberosum")

print("\nPredição para PHYSALIS\n")
predizer_transcritos(transcriptoma_physalis_fasta, rf_model,"RF",output="predicoes_physalis_rf.csv",planta="Physalis peruviana")

print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_rf.csv")







Predição para ARABIDOPSIS


TOP CANDIDATOS com probabilidade de >= {probabilidade}: pipelineY/arabidopsis_transcriptoma.fasta, MODELO: RF
                 id                                          descricao  \
2633  LQ605013.1.p1  LQ605013.1.p1 GENE.LQ605013.1~~LQ605013.1.p1  ...   
1931  JB349452.1.p1  JB349452.1.p1 GENE.JB349452.1~~JB349452.1.p1  ...   
2559  LQ603166.1.p1  LQ603166.1.p1 GENE.LQ603166.1~~LQ603166.1.p1  ...   
1857  JB347605.1.p1  JB347605.1.p1 GENE.JB347605.1~~JB347605.1.p1  ...   
1896  JB347889.1.p1  JB347889.1.p1 GENE.JB347889.1~~JB347889.1.p1  ...   
2598  LQ603450.1.p1  LQ603450.1.p1 GENE.LQ603450.1~~LQ603450.1.p1  ...   
2607  LQ603544.1.p1  LQ603544.1.p1 GENE.LQ603544.1~~LQ603544.1.p1  ...   
1870  JB347704.1.p1  JB347704.1.p1 GENE.JB347704.1~~JB347704.1.p1  ...   
2572  LQ603265.1.p1  LQ603265.1.p1 GENE.LQ603265.1~~LQ603265.1.p1  ...   
1905  JB347983.1.p1  JB347983.1.p1 GENE.JB347983.1~~JB347983.1.p1  ...   
2588  LQ603401.1.p1  LQ603401.1.p1 GENE.LQ60340

In [12]:
'''
Aplicação - Modelo SVM
'''

print("\nPredição para ARABIDOPSIS\n")
predizer_transcritos(transcriptoma_arabidopsis_fasta, svm_model,"SVM",output="predicoes_arabidopsis_svm.csv",planta="Arabidopsis thaliana")

print("\nPredição para MELONGENA\n")
predizer_transcritos(transcriptoma_melongena_fasta, svm_model,'SVM',output="predicoes_melongena_svm.csv",planta="Solanum melongena")

print("\nPredição para TUBEROSUM\n")
predizer_transcritos(transcriptoma_tuberosum_fasta, svm_model,"SVM",output="predicoes_tuberosum_svm.csv",planta="Solanum tuberosum")

print("\nPredição para PHYSALIS\n")
predizer_transcritos(transcriptoma_physalis_fasta, svm_model,"SVM",output="predicoes_physalis_svm.csv",planta="Physalis peruviana")  

print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_svm.csv")



Predição para ARABIDOPSIS


TOP CANDIDATOS com probabilidade de >= {probabilidade}: pipelineY/arabidopsis_transcriptoma.fasta, MODELO: SVM
                 id                                          descricao  \
1896  JB347889.1.p1  JB347889.1.p1 GENE.JB347889.1~~JB347889.1.p1  ...   
2598  LQ603450.1.p1  LQ603450.1.p1 GENE.LQ603450.1~~LQ603450.1.p1  ...   
2559  LQ603166.1.p1  LQ603166.1.p1 GENE.LQ603166.1~~LQ603166.1.p1  ...   
1857  JB347605.1.p1  JB347605.1.p1 GENE.JB347605.1~~JB347605.1.p1  ...   
471   AX506539.1.p1  AX506539.1.p1 GENE.AX506539.1~~AX506539.1.p1  ...   
2694  PV383300.1.p1  PV383300.1.p1 GENE.PV383300.1~~PV383300.1.p1  ...   
2572  LQ603265.1.p1  LQ603265.1.p1 GENE.LQ603265.1~~LQ603265.1.p1  ...   
2607  LQ603544.1.p1  LQ603544.1.p1 GENE.LQ603544.1~~LQ603544.1.p1  ...   
1870  JB347704.1.p1  JB347704.1.p1 GENE.JB347704.1~~JB347704.1.p1  ...   
1905  JB347983.1.p1  JB347983.1.p1 GENE.JB347983.1~~JB347983.1.p1  ...   
1886  JB347840.1.p1  JB347840.1.p1 GENE.JB3478

In [13]:
print(df_validacao)

[{'P_LOOP': 32, 'KINASE2': 25, 'RNBS_A': 13, 'RNBS_B': 2, 'RNBS_C': 6, 'GLPL': 15, 'RNBS_D': 15, 'MHD': 17, 'planta': 'Arabidopsis thaliana', 'modelo': 'XGBoost'}, {'P_LOOP': 8, 'KINASE2': 14, 'RNBS_A': 2, 'RNBS_B': 7, 'RNBS_C': 0, 'GLPL': 12, 'RNBS_D': 12, 'MHD': 2, 'planta': 'Solanum melongena', 'modelo': 'XGBoost'}, {'P_LOOP': 8, 'KINASE2': 2, 'RNBS_A': 1, 'RNBS_B': 5, 'RNBS_C': 1, 'GLPL': 7, 'RNBS_D': 8, 'MHD': 2, 'planta': 'Solanum tuberosum', 'modelo': 'XGBoost'}, {'P_LOOP': 52, 'KINASE2': 57, 'RNBS_A': 13, 'RNBS_B': 18, 'RNBS_C': 10, 'GLPL': 44, 'RNBS_D': 39, 'MHD': 17, 'planta': 'Physalis peruviana', 'modelo': 'XGBoost'}, {'P_LOOP': 17, 'KINASE2': 17, 'RNBS_A': 8, 'RNBS_B': 1, 'RNBS_C': 4, 'GLPL': 13, 'RNBS_D': 8, 'MHD': 12, 'planta': 'Arabidopsis thaliana', 'modelo': 'RF'}, {'P_LOOP': 6, 'KINASE2': 7, 'RNBS_A': 2, 'RNBS_B': 4, 'RNBS_C': 0, 'GLPL': 5, 'RNBS_D': 5, 'MHD': 2, 'planta': 'Solanum melongena', 'modelo': 'RF'}, {'P_LOOP': 2, 'KINASE2': 2, 'RNBS_A': 1, 'RNBS_B': 0, 'RN

In [14]:
df_copia = df_validacao.copy()
df_inicial = pd.DataFrame(df_validacao)

df_validacao = df_inicial.melt(
    id_vars=["planta", "modelo"], 
    var_name="motivo", 
    value_name="contagem"
)
print(df_validacao.head())

df_validacao.to_csv(f"{OUTPUT_DIR}/df_validacao.csv", index=False)

                 planta   modelo  motivo  contagem
0  Arabidopsis thaliana  XGBoost  P_LOOP        32
1     Solanum melongena  XGBoost  P_LOOP         8
2     Solanum tuberosum  XGBoost  P_LOOP         8
3    Physalis peruviana  XGBoost  P_LOOP        52
4  Arabidopsis thaliana       RF  P_LOOP        17


In [15]:
#Scatterplot
plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df_validacao,
    x="motivo",
    y="contagem",
    hue="planta",
    style="modelo",
    s=200
)
# plt.title( "Validação biológica dos motivos conservados")
plt.ylabel("Número de ocorrências")
plt.xlabel("Motivo")
plt.legend(
    bbox_to_anchor=(1.05,1),
    loc="upper left"
)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Validacao_biologica_dos_motivos_conservados.png")
plt.close()
# plt.show()

#Bubble Chart
plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df_validacao,
    x="motivo",
    y="modelo",
    hue="planta",
    size="contagem",
    sizes=(50,600)
)
# plt.title("Distribuição dos motivos por modelo e espécie")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/distribuicao_dos_motivos_por_modelo_especie.png")
plt.close()
# plt.show()


#Facetas por planta
g = sns.catplot(
    data=df_validacao,
    x="motivo",
    y="contagem",
    hue="modelo",
    col="planta",
    kind="bar",
    height=5,
    aspect=1.2
)
g.set_xticklabels(rotation=45)
plt.savefig(f"{OUTPUT_DIR}/facetas_por_planta.png")
plt.close()
# plt.show()

#por planta

#Physalis peruviana
df_physalis = df_validacao[df_validacao["planta"] == "Physalis peruviana"]
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.lineplot(
    data=df_physalis,
    x="motivo",
    y="contagem",
    hue="modelo",
    style="modelo",
    markers=True,     # Ativa os marcadores nos pontos
    dashes=False,     # Mantém as linhas contínuas
    linewidth=2,      # Espessura da linha
    markersize=10     # Tamanho dos pontos de dispersão
)

# plt.title("Comparação dos Modelos para Physalis peruviana", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Motivos Conservados", fontsize=12, labelpad=10)
plt.ylabel("Número de Ocorrências (Contagem)", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Modelos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparacao_modelos_physalis.png", dpi=300)
plt.close()

#Solanum melongena
df_melongena = df_validacao[df_validacao["planta"] == "Solanum melongena"]
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.lineplot(
    data=df_melongena,
    x="motivo",
    y="contagem",
    hue="modelo",
    style="modelo",
    markers=True,     # Ativa os marcadores nos pontos
    dashes=False,     # Mantém as linhas contínuas
    linewidth=2,      # Espessura da linha
    markersize=10     # Tamanho dos pontos de dispersão
)

# plt.title("Comparação dos Modelos para Solanum melongena", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Motivos Conservados", fontsize=12, labelpad=10)
plt.ylabel("Número de Ocorrências (Contagem)", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Modelos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparacao_modelos_melongena.png", dpi=300)
plt.close()

#Solanum tuberosum
df_tuberosum = df_validacao[df_validacao["planta"] == "Solanum tuberosum"]
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.lineplot(
    data=df_tuberosum,
    x="motivo",
    y="contagem",
    hue="modelo",
    style="modelo",
    markers=True,     # Ativa os marcadores nos pontos
    dashes=False,     # Mantém as linhas contínuas
    linewidth=2,      # Espessura da linha
    markersize=10     # Tamanho dos pontos de dispersão
)

# plt.title("Comparação dos Modelos para Solanum tuberosum", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Motivos Conservados", fontsize=12, labelpad=10)
plt.ylabel("Número de Ocorrências (Contagem)", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Modelos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparacao_modelos_tuberosum.png", dpi=300)
plt.close()


#Heatmap
df_validacao["grupo"] = (
    df_validacao["planta"]
    + "_"
    + df_validacao["modelo"]
)

pivot = df_validacao.pivot_table(
    index="motivo",
    columns="grupo",
    values="contagem",
    fill_value=0
)


plt.figure(figsize=(14, 8))
sns.heatmap(
    pivot,
    annot=True,
    fmt="g",
    cmap="YlGnBu",
    linewidths=0.5,
    cbar_kws={'label': 'Contagem'} # Nomeia a barra de cores lateral
)

# plt.title("Distribuição dos Motivos Conservados por Planta e Modelo", fontsize=14, fontweight="bold", pad=15)
plt.ylabel("Motivo", fontsize=12)
plt.xlabel("Grupo (Planta & Modelo)", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/heatmap_distribuicao_dos_motivos_conservados.png",dpi=300)
plt.close()
# plt.show()


print("========= Fim do Pipeline ============")

========= Fim do Pipeline ============
